In [0]:
%sql
use catalog deltalake_catalog;

In [0]:

%sql
DROP TABLE IF EXISTS orders_managed;

CREATE OR REPLACE TABLE orders_managed (
  order_id BIGINT,
  sku STRING,
  product_name STRING,
  product_category STRING,
  qty INT,
  unit_price DECIMAL(10,2)
)
USING DELTA;


In [0]:
%sql
describe detail orders_managed;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,7d0af3ad-8f2e-4657-8dd1-c1d8b6b8e6b7,deltalake_catalog.default.orders_managed,null,abfss://unitycatalog@ttmystorageaccount001.dfs.core.windows.net/catalog/__unitystorage/catalogs/758280e5-e8ae-48b0-840c-9f10d52cc821/tables/2b2bc737-72e4-480c-ab87-29ac3a560fe0,2025-09-26T11:21:13.481Z,2025-09-26T11:21:13.000Z,List(),List(),0,0,"Map(collation -> UTF8_BINARY, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import DecimalType
from pyspark.sql.functions import col
import time


N = 100

for i in range(N):
    row = Row(order_id=int(i+1),
              sku=f"SKU-{(i%10)+1}",
              product_name=f"Product-{(i%50)+1}",
              product_category=f"Category-{(i%5)+1}",
              qty=(i % 10) + 1,
              unit_price=round(10.0 + (i%7)*1.5, 2))
    
    df = spark.createDataFrame([row])

    df = df.withColumn("qty", col("qty").cast("int")).withColumn("unit_price", col("unit_price").cast(DecimalType(10,2)))
    
    df.write.format("delta").mode("append").saveAsTable("orders_managed")
    # optional tiny sleep to mimic real small-batch writes
    # time.sleep(0.02)

print(f"Inserted {N} tiny writes.")

Inserted 100 tiny writes.


In [0]:
%sql
select * from orders_managed;

In [0]:
%sql
select * from orders_managed where order_id = 56;

In [0]:
query = "select AVG(unit_price) as avg_price from orders_managed"
res = spark.sql(query).collect()
print(res)

[Row(avg_price=Decimal('14.425000'))]


In [0]:
%sql
DESCRIBE HISTORY orders_managed;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
103,2025-09-26T11:23:13.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835752),0926-100404-x5jzm2ub-v2n,102,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1726)",null,Databricks-Runtime/17.1.x-photon-scala2.13
102,2025-09-26T11:23:12.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835752),0926-100404-x5jzm2ub-v2n,101,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1721)",null,Databricks-Runtime/17.1.x-photon-scala2.13
101,2025-09-26T11:23:11.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835752),0926-100404-x5jzm2ub-v2n,100,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1720)",null,Databricks-Runtime/17.1.x-photon-scala2.13
100,2025-09-26T11:23:10.001Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835752),0926-100404-x5jzm2ub-v2n,99,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1721)",null,Databricks-Runtime/17.1.x-photon-scala2.13
99,2025-09-26T11:23:10.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835752),0926-100404-x5jzm2ub-v2n,98,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1721)",null,Databricks-Runtime/17.1.x-photon-scala2.13
98,2025-09-26T11:23:09.000Z,147711536460494,sumit.trendytech03@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2999395735835752),0926-100404-x5jzm2ub-v2n,96,SnapshotIsolation,false,"Map(numRemovedFiles -> 32, numRemovedBytes -> 56207, p25FileSize -> 3159, numDeletionVectorsRemoved -> 0, conflictDetectionTimeMs -> 13, minFileSize -> 3159, numAddedFiles -> 1, maxFileSize -> 3159, p75FileSize -> 3159, p50FileSize -> 3159, numAddedBytes -> 3159)",null,Databricks-Runtime/17.1.x-photon-scala2.13
97,2025-09-26T11:23:08.001Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835752),0926-100404-x5jzm2ub-v2n,96,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1720)",null,Databricks-Runtime/17.1.x-photon-scala2.13
96,2025-09-26T11:23:08.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835752),0926-100404-x5jzm2ub-v2n,95,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1721)",null,Databricks-Runtime/17.1.x-photon-scala2.13
95,2025-09-26T11:23:07.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835752),0926-100404-x5jzm2ub-v2n,94,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1721)",null,Databricks-Runtime/17.1.x-photon-scala2.13
94,2025-09-26T11:23:06.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835752),0926-100404-x5jzm2ub-v2n,93,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1721)",null,Databricks-Runtime/17.1.x-photon-scala2.13


In [0]:
%sql
OPTIMIZE orders_managed;

In [0]:
query = "select AVG(unit_price) as avg_price from orders_managed"
res = spark.sql(query).collect()
print(res)